# Treinamento Híbrido: Random Forest + PCA + MLP\n
Este notebook utiliza Random Forest para limpar variáveis inúteis, PCA para remover colinearidade (dados repetidos), e um MLP profundo para o treinamento final.

In [34]:
import os
import h5py
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

## 1. Carregando os Dados e Gerando Features Temporais (RAM Otimizada)

In [35]:
# Carregar arquivo
filename = 'data/N-CMAPSS_DS02-006.h5'
with h5py.File(filename, 'r') as hdf:
    W_dev = np.array(hdf.get('W_dev'))
    X_s_dev = np.array(hdf.get('X_s_dev'))
    Y_dev = np.array(hdf.get('Y_dev'))
    A_dev = np.array(hdf.get('A_dev'))
    
    W_test = np.array(hdf.get('W_test'))
    X_s_test = np.array(hdf.get('X_s_test'))
    Y_test = np.array(hdf.get('Y_test'))
    A_test = np.array(hdf.get('A_test'))

def create_temporal_features_safe(W, X_s, Y, A, window=40):
    A_features = A[:, 1:].astype('float32') # Traz o ciclo, classe de voo e hs
    matriz_base = np.concatenate((W, X_s, A_features), axis=1).astype('float32')
    
    df = pd.DataFrame(matriz_base)
    df['unit'] = A[:, 0].astype('float32')
    df['RUL'] = Y.flatten().astype('float32')
    
    print("Calculando médias e variâncias (Modo Economia de RAM)...")
    df_mean = df.groupby('unit').rolling(window=window, min_periods=1).mean().reset_index(level=0, drop=True).sort_index().astype('float32')
    df_var = df.groupby('unit').rolling(window=window, min_periods=1).var().fillna(0).reset_index(level=0, drop=True).sort_index().astype('float32')
    
    df_mean = df_mean[df.columns[:-2]]
    df_var = df_var[df.columns[:-2]]
    df_raw = pd.DataFrame(matriz_base)
    
    X_temporal = pd.concat([df_raw, df_mean, df_var], axis=1).values.astype('float32')
    y_labels = df['RUL'].values.astype('float32')
    
    del df, df_mean, df_var, df_raw, matriz_base
    gc.collect()
    
    return X_temporal, y_labels

In [36]:
print("Processando Treino...")
X_train_raw, y_train_full = create_temporal_features_safe(W_dev, X_s_dev, Y_dev, A_dev, window=20)

print("Processando Teste...")
X_test_raw, y_test = create_temporal_features_safe(W_test, X_s_test, Y_test, A_test, window=20)

# Deleta dados brutos antigos para poupar RAM
del W_dev, X_s_dev, Y_dev, A_dev
del W_test, X_s_test, Y_test, A_test
gc.collect()

Processando Treino...
Calculando médias e variâncias (Modo Economia de RAM)...
Processando Teste...
Calculando médias e variâncias (Modo Economia de RAM)...


0

## 2. Escalonamento e Random Forest (Seleção das Melhores Features)\n
Aqui usamos Random Forest para achar as colunas que realmente importam para prever RUL.

In [37]:
# Escalonamento Inicial
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw).astype('float32')
X_test_scaled = scaler.transform(X_test_raw).astype('float32')

del X_train_raw, X_test_raw
gc.collect()



0

In [38]:
print("Treinando Random Forest para achar as melhores features...")
amostra = int(len(X_train_scaled) * 0.05)

rf = RandomForestRegressor(n_estimators=30, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled[:amostra], y_train_full[:amostra])

# O SelectFromModel corta automaticamente as variáveis inúteis
selector = SelectFromModel(rf, prefit=True, threshold=-np.inf, max_features=25)
X_train_selected = selector.transform(X_train_scaled)
X_test_selected = selector.transform(X_test_scaled)

print(f"Colunas originais: {X_train_scaled.shape[1]}")
print(f"Colunas mantidas pelo Random Forest: {X_train_selected.shape[1]}")

del X_train_scaled, X_test_scaled
gc.collect()

Treinando Random Forest para achar as melhores features...
Colunas originais: 63
Colunas mantidas pelo Random Forest: 25


72

## 3. PCA (Redução de Dimensionalidade)\n
Pegamos as colunas aprovadas pela árvore e passamos no PCA para garantir que não haja multicolinearidade (retemos 98% da variância).

In [39]:
# EMBARALHANDO 100% dos dados mantendo os 25 sensores puros da Random Forest
X_train_shuffled, _, y_train_shuffled, _ = train_test_split(
    X_train_selected, y_train_full, test_size=0.0001, random_state=42
)

del X_train_selected, y_train_full
gc.collect()

print(f"Colunas que vão entrar puras na Rede Neural: {X_train_shuffled.shape[1]}")


Colunas que vão entrar puras na Rede Neural: 25


## 4. Treinamento da MLP Profunda\n
Com os dados totalmente limpos, a rede neural não sofre de confusão matemática.

In [46]:
final_model = Sequential()
final_model.add(tf.keras.layers.Input(shape=(X_train_shuffled.shape[1],)))

final_model.add(Dense(units=128, activation='relu'))
final_model.add(BatchNormalization())
final_model.add(Dropout(0.2))

final_model.add(Dense(units=64, activation='relu'))
final_model.add(BatchNormalization())
final_model.add(Dropout(0.2))

final_model.add(Dense(units=32, activation='relu'))
final_model.add(BatchNormalization())

final_model.add(Dense(1, activation='linear'))

final_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='mse', metrics=['mae'])


In [47]:
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5, verbose=1)
stop_early = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)

In [48]:


print("Iniciando Treinamento Oficial Híbrido...")
history = final_model.fit(
    X_train_shuffled, y_train_shuffled,
    validation_split=0.2, 
    epochs=100, 
    batch_size=4096, 
    callbacks=[stop_early, reduce_lr],
    verbose=1
)

Iniciando Treinamento Oficial Híbrido...
Epoch 1/100


I0000 00:00:1789434960.002587  181414 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_2295164__.30


1016/1028 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1167.0827 - mae: 30.9701

I0000 00:00:1789434965.554496  181417 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_2295164__.30


1028/1028 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - loss: 669.7085 - mae: 21.7870 - val_loss: 48.1774 - val_mae: 5.0545 - learning_rate: 0.0010
Epoch 2/100
1028/1028 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 37.6603 - mae: 4.5389 - val_loss: 33.2905 - val_mae: 4.0884 - learning_rate: 0.0010
Epoch 3/100
1028/1028 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 33.9390 - mae: 4.2382 - val_loss: 28.5878 - val_mae: 3.7538 - learning_rate: 0.0010
Epoch 4/100
1028/1028 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 31.5343 - mae: 4.0359 - val_loss: 25.8299 - val_mae: 3.5500 - learning_rate: 0.0010
Epoch 5/100
1028/1028 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 29.7058 - mae: 3.8842 - val_loss: 24.8901 - val_mae: 3.4346 - learning_rate: 0.0010
Epoch 6/100
1028/1028 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 28.6176 - mae: 3.7913 - val_loss: 23.7566 - val_mae: 3.3109 - learning_rate: 0.0010
Epoch 7/100
1028/1028 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 27.8604 - mae: 3.7263 - val_loss: 23.8269 - val_mae: 3.4248 - le

## 5. Avaliação Final e Exportação

In [51]:
y_pred = final_model.predict(X_test_selected)

39180/39180 ━━━━━━━━━━━━━━━━━━━━ 34s 859us/step


In [53]:

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"✅ MAE no Teste: {mae:.2f}")
print(f"✅ RMSE no Teste: {rmse:.2f}")
print(f"✅ R² Score Final: {r2:.4f}")

✅ MAE no Teste: 7.02
✅ RMSE no Teste: 9.90
✅ R² Score Final: 0.7276


In [ ]:
import joblib
final_model.save('data/modelo_preditivo_hibrido_rul.keras')
joblib.dump(scaler, 'data/scaler.pkl')
joblib.dump(selector, 'data/rf_selector.pkl')
joblib.dump(pca, 'data/pca_transform.pkl')
print("Modelos e transformadores exportados com sucesso!")